# Aligned Development Feature Ablation

This notebook documents the aligned comparison of six and twelve stacking meta-features. It repeats the development-stage XGBoost comparison across the fixed seeds to isolate the contribution of confidence, entropy, and base-disagreement features.


## Release Boundary

This restricted code-inspection copy retains the analytical logic but does not include Banjarbaru review data, row-level predictions, manual-audit records, model checkpoints, or private run artifacts. The restricted path placeholder documents the authorized environment and is not an executable public-data contract.


### Configure the feature-ablation protocol

Imports the XGBoost and metric utilities, defines the restricted project-root placeholder, identifies the private base-model and stacking inputs, and declares the six probability features, six additional uncertainty features, and expected seed sizes.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import uuid
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier


PROJECT_ROOT = Path('RESTRICTED_PROJECT_ROOT')
ROUND1_DIR = PROJECT_ROOT / 'journal_round1'
R1_06_ROOT = ROUND1_DIR / 'artifacts' / 'r1_06_repeated_seed_base_oof_v2'
R1_07_ROOT = ROUND1_DIR / 'artifacts' / 'r1_07_revised_stacking_calibration_v1'
OUTPUT_DIR = ROUND1_DIR / 'reports' / 'aligned_feature_ablation' / 'r1_10'


EXPECTED_SEEDS = [42, 52, 62]
LABEL_ORDER = ['negatif', 'netral', 'positif']
SIX_FEATURES = [
    'svm_prob_negatif', 'svm_prob_netral', 'svm_prob_positif',
    'bert_prob_negatif', 'bert_prob_netral', 'bert_prob_positif',
]
TWELVE_FEATURES = SIX_FEATURES + [
    'svm_confidence', 'bert_confidence', 'svm_entropy', 'bert_entropy',
    'pred_disagreement', 'prob_disagreement',
]
EXPECTED_OOF_N = 4673
EXPECTED_VALIDATION_N = 1170

if not R1_06_ROOT.is_dir() or not R1_07_ROOT.is_dir():
    raise FileNotFoundError('Required base-model or stacking artifacts are missing.')

print('Restricted project root placeholder:', PROJECT_ROOT)
print('Base-model source:', R1_06_ROOT)
print('Stacking source:', R1_07_ROOT)
print('Output:', OUTPUT_DIR)


### Define persistence, metric, and paired-comparison utilities

Defines checksum and atomic-write helpers, XGBoost construction, probability validation, performance metrics, McNemar calculations, and Holm adjustment for the aligned ablation comparison.


In [ ]:
def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_csv(path: Path, frame: pd.DataFrame) -> None:
    temporary = path.with_name(f'.{path.name}.{uuid.uuid4().hex}.tmp')
    frame.to_csv(temporary, index=False, encoding='utf-8')
    os.replace(temporary, path)


def atomic_json(path: Path, payload: dict) -> None:
    temporary = path.with_name(f'.{path.name}.{uuid.uuid4().hex}.tmp')
    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=True, default=float), encoding='utf-8')
    os.replace(temporary, path)


def exact_mcnemar(twelve_correct: np.ndarray, six_correct: np.ndarray) -> dict:
    twelve_only = int(np.sum(twelve_correct & ~six_correct))
    six_only = int(np.sum(~twelve_correct & six_correct))
    both_correct = int(np.sum(twelve_correct & six_correct))
    both_incorrect = int(np.sum(~twelve_correct & ~six_correct))
    discordant = twelve_only + six_only
    raw_p = 1.0 if discordant == 0 else float(binomtest(twelve_only, n=discordant, p=0.5, alternative='two-sided').pvalue)
    return {
        'both_correct': both_correct, 'twelve_only_correct': twelve_only,
        'six_only_correct': six_only, 'both_incorrect': both_incorrect,
        'discordant_n': discordant, 'exact_raw_p_value': raw_p,
    }


def holm_adjust(frame: pd.DataFrame, p_column: str) -> pd.DataFrame:
    output = frame.copy()
    ordered = output.sort_values(p_column, kind='mergesort').copy()
    total = len(ordered)
    running = 0.0
    adjusted = []
    thresholds = []
    for rank, raw_p in enumerate(ordered[p_column].to_numpy(float), start=1):
        running = max(running, min(1.0, raw_p * (total - rank + 1)))
        adjusted.append(running)
        thresholds.append(0.05 / (total - rank + 1))
    ordered['holm_rank'] = np.arange(1, total + 1)
    ordered['holm_adjusted_p_value'] = adjusted
    ordered['holm_step_threshold'] = thresholds
    ordered['holm_reject_alpha_0_05'] = ordered['holm_adjusted_p_value'] < 0.05
    return ordered.sort_index()


def validate_meta_frame(frame: pd.DataFrame, seed: int, split_name: str, expected_n: int) -> pd.DataFrame:
    required = {'row_id', 'y_silver', *TWELVE_FEATURES}
    missing = sorted(required.difference(frame.columns))
    if missing:
        raise ValueError(f'Seed {seed}/{split_name}: missing columns {missing}.')
    if len(frame) != expected_n or frame['row_id'].duplicated().any():
        raise ValueError(f'Seed {seed}/{split_name}: row count or row_id uniqueness check failed.')
    if not set(frame['y_silver'].unique()).issubset({0, 1, 2}):
        raise ValueError(f'Seed {seed}/{split_name}: y_silver is outside the three-class label space.')
    values = frame[TWELVE_FEATURES].to_numpy(float)
    if not np.isfinite(values).all():
        raise ValueError(f'Seed {seed}/{split_name}: non-finite meta-feature found.')
    return frame.sort_values('row_id').reset_index(drop=True)


### Load and validate aligned seed inputs

Checks the base-model protocol, OOF and validation artifact structure, seed compatibility, XGBoost recipe consistency, and input checksums before feature configurations are compared.


In [ ]:
source_protocol = json.loads((R1_06_ROOT / 'protocol_v2.json').read_text(encoding='utf-8'))
if source_protocol.get('planned_seeds') != EXPECTED_SEEDS:
    raise RuntimeError('Base-model seeds do not match the fixed feature-ablation contract.')

seed_inputs = {}
input_hashes = {}
xgb_configs = []
for seed in EXPECTED_SEEDS:
    seed06 = R1_06_ROOT / f'seed_{seed}'
    seed07 = R1_07_ROOT / f'seed_{seed}'
    oof_path = seed06 / 'oof_train_meta_features.csv'
    validation_path = seed06 / 'validation_meta_features.csv'
    manifest07_path = seed07 / 'run_manifest.json'
    for path in [oof_path, validation_path, manifest07_path]:
        if not path.is_file():
            raise FileNotFoundError(path)

    oof = validate_meta_frame(pd.read_csv(oof_path), seed, 'oof_train', EXPECTED_OOF_N)
    validation = validate_meta_frame(pd.read_csv(validation_path), seed, 'validation', EXPECTED_VALIDATION_N)

    manifest07 = json.loads(manifest07_path.read_text(encoding='utf-8'))
    config = manifest07['protocol']['xgboost_config']
    xgb_configs.append(config)
    seed_inputs[seed] = {'oof': oof, 'validation': validation, 'xgb_config': config}
    input_hashes[str(seed)] = {
        'oof_train_meta_features_sha256': sha256_file(oof_path),
        'validation_meta_features_sha256': sha256_file(validation_path),
        'r1_07_run_manifest_sha256': sha256_file(manifest07_path),
    }

if any(config != xgb_configs[0] for config in xgb_configs[1:]):
    raise RuntimeError('The stacking XGBoost recipe differs across seeds.')
XGB_CONFIG = xgb_configs[0].copy()

schema = {
    'six_feature_names': SIX_FEATURES,
    'twelve_feature_names': TWELVE_FEATURES,
    'six_is_subset_of_twelve': set(SIX_FEATURES).issubset(TWELVE_FEATURES),
    'source_r1_06_protocol_hash': source_protocol.get('protocol_hash'),
    'source_xgboost_config': XGB_CONFIG,
}
if not schema['six_is_subset_of_twelve'] or len(TWELVE_FEATURES) != 12:
    raise RuntimeError('Feature schema contract is invalid.')
atomic_json(OUTPUT_DIR / 'feature_schema.json', schema)
print('Validated seeds:', list(seed_inputs))
print('XGBoost recipe:', XGB_CONFIG)


### Compare six and twelve meta-feature configurations

Fits the same XGBoost recipe with probability-only and full uncertainty-aware meta-features for every seed, evaluates OOF and validation performance, and calculates aligned paired comparisons.


In [ ]:
metric_rows = []
prediction_rows = []
mcnemar_rows = []

for seed, inputs in seed_inputs.items():
    oof = inputs['oof']
    validation = inputs['validation']
    y_train = oof['y_silver'].to_numpy(int)
    y_validation = validation['y_silver'].to_numpy(int)
    predicted = {}

    for configuration, features in [('six_feature', SIX_FEATURES), ('twelve_feature', TWELVE_FEATURES)]:
        classifier = XGBClassifier(random_state=seed, **XGB_CONFIG)
        classifier.fit(oof[features], y_train)
        probability = classifier.predict_proba(validation[features])
        hard_prediction = probability.argmax(axis=1).astype(int)
        predicted[configuration] = hard_prediction
        metric_rows.append({
            'seed': seed, 'configuration': configuration, 'feature_count': len(features),
            'reference_target': 'validation_silver', 'n': len(y_validation),
            'accuracy': float(accuracy_score(y_validation, hard_prediction)),
            'macro_f1': float(f1_score(y_validation, hard_prediction, average='macro', zero_division=0)),
            'weighted_f1': float(f1_score(y_validation, hard_prediction, average='weighted', zero_division=0)),
        })
        prediction_rows.append(pd.DataFrame({
            'seed': seed, 'row_id': validation['row_id'].to_numpy(), 'y_silver': y_validation,
            'configuration': configuration, 'feature_count': len(features),
            'prediction': hard_prediction, 'confidence': probability.max(axis=1),
            'prob_negatif': probability[:, 0], 'prob_netral': probability[:, 1], 'prob_positif': probability[:, 2],
        }))

    twelve_correct = predicted['twelve_feature'] == y_validation
    six_correct = predicted['six_feature'] == y_validation
    comparison = exact_mcnemar(twelve_correct, six_correct)
    mcnemar_rows.append({
        'seed': seed, 'reference_target': 'validation_silver', 'contrast': 'twelve_feature versus six_feature',
        'n': len(y_validation),
        'delta_accuracy_twelve_minus_six': float(twelve_correct.mean() - six_correct.mean()),
        **comparison,
    })

metrics_per_seed = pd.DataFrame(metric_rows).sort_values(['seed', 'configuration']).reset_index(drop=True)
predictions_per_seed = pd.concat(prediction_rows, ignore_index=True).sort_values(['seed', 'configuration', 'row_id']).reset_index(drop=True)
mcnemar_per_seed = holm_adjust(pd.DataFrame(mcnemar_rows), 'exact_raw_p_value').sort_values('seed').reset_index(drop=True)

if len(metrics_per_seed) != 6 or len(predictions_per_seed) != 2 * len(EXPECTED_SEEDS) * EXPECTED_VALIDATION_N:
    raise RuntimeError('Unexpected feature-ablation output row count.')
if not predictions_per_seed.groupby(['seed', 'configuration'])['row_id'].nunique().eq(EXPECTED_VALIDATION_N).all():
    raise RuntimeError('Validation predictions are incomplete.')

print(metrics_per_seed.to_string(index=False))
print(mcnemar_per_seed.to_string(index=False))


### Write aggregate ablation outputs

Writes per-seed metrics, repeated-seed summaries, McNemar results, and the semantic completion manifest. The notebook reports contribution of added features rather than selecting a new final model.


In [ ]:
summary = (
    metrics_per_seed.groupby(['configuration', 'feature_count', 'reference_target'], as_index=False)
    .agg(
        seed_n=('seed', 'nunique'),
        accuracy_mean=('accuracy', 'mean'), accuracy_sample_sd=('accuracy', 'std'),
        macro_f1_mean=('macro_f1', 'mean'), macro_f1_sample_sd=('macro_f1', 'std'),
        weighted_f1_mean=('weighted_f1', 'mean'), weighted_f1_sample_sd=('weighted_f1', 'std'),
    )
    .sort_values('feature_count').reset_index(drop=True)
)

for filename, frame in {
    'feature_ablation_metrics_per_seed.csv': metrics_per_seed,
    'feature_ablation_repeated_seed_summary.csv': summary,
    'feature_ablation_validation_predictions_per_seed.csv': predictions_per_seed,
    'feature_ablation_mcnemar_holm_per_seed.csv': mcnemar_per_seed,
}.items():
    atomic_csv(OUTPUT_DIR / filename, frame)

manifest = {
    'status': 'completed',
    'completed_at_utc': utc_now(),
    'analysis_scope': 'validation-only aligned development-stage feature ablation',
    'seeds': EXPECTED_SEEDS,
    'reference_target': 'LSTM-derived silver labels on validation only',
    'locked_test_labels_read': False,
    'base_models_retrained': False,
    'calibration_fitted': False,
    'hyperparameter_search_performed': False,
    'feature_schema': schema,
    'holm_family_size': int(len(mcnemar_per_seed)),
    'input_hashes': input_hashes,
    'output_checksums': {filename: sha256_file(OUTPUT_DIR / filename) for filename in [
        'feature_schema.json', 'feature_ablation_metrics_per_seed.csv',
        'feature_ablation_repeated_seed_summary.csv',
        'feature_ablation_validation_predictions_per_seed.csv',
        'feature_ablation_mcnemar_holm_per_seed.csv',
    ]},
    'interpretation_guardrail': 'Do not treat this validation-only development comparison as final locked-test superiority or a sixth final model.',
}
atomic_json(OUTPUT_DIR / 'analysis_manifest.json', manifest)
atomic_json(OUTPUT_DIR / 'run_state.json', {
    'status': 'completed', 'completed_at_utc': utc_now(),
    'completed_stages': ['input_validation', 'schema_validation', 'xgboost_fitting', 'validation_evaluation', 'holm_mcnemar', 'finalization'],
})

print('Feature ablation completed:', OUTPUT_DIR)
print(summary.to_string(index=False))
print('Interpret this as validation-only development evidence, not final-model superiority.')
